# RMSProp Simulations - Non-Convex

## Parameters

In [0]:
from solvers import RMSPropMomentum, NonlocalSolverMomentumRMSProp
from sklearn.model_selection import ParameterGrid
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os
import jax 
import jax.numpy as jnp

# Hyperparameter grid:
# - Two learning rates (lr)
# - Three RMSProp momentum factors (beta)
param_grid = {'lr': [0.1, 0.01], 'beta': [0.0, 0.9, 0.99]}
n_learning_rates = len(param_grid['lr'])

# All combinations of the grid as a list of dicts, e.g.
# [{'lr': 0.1, 'beta': 0.0}, {'lr': 0.1, 'beta': 0.9}, ...]
param_list = list(ParameterGrid(param_grid))

# Problem setup:
# dL(y) is the gradient of the loss w.r.t. the scalar parameter y:
# here dL(y) = y (y^2 − 1), i.e., derivative of (1/4)(y^2 − 1)^2.
dL = lambda y: y * (y**2 - 1)
f = lambda x, y: 0.0

# Create the output folder for figures if it doesn't exist yet
figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)

## RMSProp - Discrete

In [0]:
# --- RMSProp | Save separate PNGs per initial condition ---
inits = [0.1, 0, -0.1]

for theta_initial in inits:
    # Create subplots for THIS initial condition
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Iterate over each learning rate
    for i, lr in enumerate(param_grid['lr']):

        # Filter parameter sets for this lr
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Number of epochs as a function of lr
        if lr == 0.1:
            epochs = 15
        elif lr == 0.01:
            epochs = 100
            
        # Run RMSProp simulations for all beta values at this lr
        for params in filtered_params:
            print(f'\nRMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver = RMSPropMomentum(dL=dL, lr=lr, beta=params['beta'], epochs=epochs)
            solver.solve(theta_initial=theta_initial)

            label = f"beta={params['beta']}"

            # θ_k trajectory
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v_k trajectory (squared gradients / second moment)
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # Layouts for THIS initial condition
    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectories for the RMSProp Optimizer — theta_initial={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_theta.update_xaxes(title_text="k")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta_k")

    fig_v.update_layout(
        title_text=f'Squared gradients convergence trajectories for the RMSProp Optimizer — theta_initial={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_v.update_xaxes(title_text="k")
    fig_v.update_yaxes(title_text="v_k")

    # File suffix (avoid dots in the number)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Guardar PNGs para ESTA condición inicial
    fig_theta.write_image(os.path.join(figures_dir, f"rmsprop_theta_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"rmsprop_v_ncvx_{suffix}.png"))

    print(f"Figuras guardadas (theta_initial={theta_initial}) en la carpeta '{figures_dir}'")

## Nonlocal RMSProp

In [0]:
# --- Nonlocal Continuous RMSProp | Save separate PNGs per initial condition ---
inits = [0.1, 0, -0.1]

for theta_initial in inits:
    # 1) Create subplots for this initial condition
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Figure titles (specific to this theta_initial)
    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectories for the first-order nonlocal continuous RMSProp — theta_initial={theta_initial}'
    )
    fig_v.update_layout(
        title_text=f'v over time for the first-order nonlocal continuous RMSProp — theta_initial={theta_initial}'
    )

    # 2) Sweep over learning rates 
    for i, lr in enumerate(param_grid['lr']):
        # Filter all parameter configs for this lr
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs as a function of lr
        if lr == 0.1:
            epochs = 15
        elif lr == 0.01:
            epochs = 100

        # Continuous span for the solver: start near 0 to avoid degeneracy
        t = [1e-12, epochs * lr]

        # 3) Run simulations for each configuration at this lr
        for params in filtered_params:
            print(f'\nNonlocal Continuous RMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver = NonlocalSolverMomentumRMSProp(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], beta=params['beta']
            )
            t_values, y_values = solver.solve()

            label = f"beta={params['beta']}"

            # θ(t) trajectory (x-axis scaled to t/alpha so it aligns with discrete k)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v(t) recorded by the solver: columns [t, v]
            denominators = np.asarray(solver._last_v)
            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # 4) Axes and sizes (for THIS θ0)
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta(t)")
    fig_theta.update_layout(width=1500, height=600)

    fig_v.update_xaxes(title_text="t/alpha")
    fig_v.update_yaxes(title_text="v(t)")
    fig_v.update_layout(width=1500, height=600)

    # 5) File suffix per initial condition (avoid dots in the number)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Save PNGs for THIS initial condition
    fig_theta.write_image(os.path.join(figures_dir, f"nonlocal_rmsprop_theta_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"nonlocal_rmsprop_v_ncvx_{suffix}.png"))

    print(f"Saved figures (theta_initial={theta_initial}) in the folder '{figures_dir}'")


## Both Models Together

In [0]:
# --- RMSProp vs. Nonlocal Continuous RMSProp
# --- Separate PNGs per initial condition theta_initial ---

config_colors = {
    (0.0): 'blue',
    (0.9): 'green',
    (0.99): 'red'
}

inits = [0.1, 0, -0.1]

for theta_initial in inits:
    # Create figures for THIS initial condition
    fig_theta = make_subplots(
        rows=1,
        cols=2,  # one column per lr (assumes 2 learning rates)
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1,
        cols=2,  # one column per lr (assumes 2 learning rates)
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Sweep over learning rates
    for i, lr in enumerate(param_grid['lr']):

        # Filter parameter sets using this lr
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs based on lr
        if lr == 0.1:
            epochs = 15
        elif lr == 0.01:
            epochs = 100

        # Continuous-time span for the nonlocal solver (avoid t=0 degeneracy)
        t = [1e-12, epochs * lr]

        # -------- Discrete RMSProp --------
        for params in filtered_params:
            color = config_colors[params['beta']]
            print(f'\nRMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver = RMSPropMomentum(dL=dL, lr=lr, beta=params['beta'], epochs=epochs)
            solver.solve(theta_initial=theta_initial)

            # θ_k
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.05), color=color),
                name=f'RMSProp beta={params["beta"]}',
                legendgroup=f'RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v_k (squared gradients)
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.05), color=color),
                name=f'RMSProp beta={params["beta"]}',
                legendgroup=f'RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

        # ---- Nonlocal continuous-time RMSProp ----
        for params in filtered_params:
            color = config_colors[params['beta']]
            print(f'\nNonlocal Continuous RMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver_nonlocal = NonlocalSolverMomentumRMSProp(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], beta=params['beta']
            )
            t_values, y_values = solver_nonlocal.solve()

            # θ(t)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal RMSProp beta={params["beta"]}',
                legendgroup=f'Nonlocal RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v(t) (saved by the solver; columns [t, v])
            denominators = np.asarray(solver_nonlocal._last_v)
            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal RMSProp beta={params["beta"]}',
                legendgroup=f'Nonlocal RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # Layouts for this initial condition
    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectories for Nonlocal Continuous RMSProp — theta_initial={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta values")

    fig_v.update_layout(
        title_text=f'Squared gradients (v) trajectories for Nonlocal Continuous RMSProp — theta_initial={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_v.update_xaxes(title_text="t/alpha")
    fig_v.update_yaxes(tickformat=".2f", title_text="v values")

    # Filename suffix (avoid dots in the number)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Save PNGs for this initial condition
    fig_theta.write_image(os.path.join(figures_dir, f"rmsprop_vs_nonlocal_theta_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"rmsprop_vs_nonlocal_v_{suffix}.png"))

    print(f"Saved figures (theta_initial={theta_initial}) in the folder '{figures_dir}'")
